# 03 Validation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# from src.utils import load_aoi, make_tiles

# # aoi = load_aoi(path="data/01_raw/uga-nakamiro/aoi/nakamiro_aoi.geojson", crs_out="EPSG:3857")
# aoi = load_aoi(path="data/01_raw/ukr-pulyny/aoi/pulyny_aoi.geojson", crs_out="EPSG:3857")
# tiles = make_tiles(aoi, 1000)
# print(f"Tiles: {len(tiles)}")

In [3]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation")
CONFIG_PATH  = PROJECT_ROOT / "configs/validation_configs.yaml"
TRACKER_PATH = PROJECT_ROOT / "data/02_interim/aoi_tracker.csv"

AOI_FILTER = None
# AOI_FILTER = ["ant-curacao", "ssd-juba"]

SKIP_EXISTING = True   # True = skip AOIs whose output parquets already exist
DRY_RUN       = False   # True = print plan only

In [4]:
import sys, os
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [5]:
import logging
from src.validator import load_config, load_tracker, process_aoi, already_done

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
    
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)


cfg        = load_config(CONFIG_PATH)
root       = PROJECT_ROOT
tracker_df = load_tracker(TRACKER_PATH, aoi_filter=AOI_FILTER)

print(f"AOIs queued: {len(tracker_df)}")
display(tracker_df[["dataset_folder_name", "aoi_file_name", "reference_file_name"]])

00:22:12  INFO      Tracker: 56 AOI rows selected for processing.


AOIs queued: 56


,dataset_folder_name,aoi_file_name,reference_file_name
0,ant-curacao,curacao_aoi.geojson,curacao_hotosm.geojson
1,blz-burrell-boom,burrell-boom_aoi.geojson,burrell-boom_hotosm.geojson
2,bra-nova-sussuarana,nova-sussuarana_aoi.geojson,nova-sussuarana_hotosm.geojson
3,col-san-antonio-de-prado,san-antonio-de-prado_aoi.geojson,san-antonio-de-prado_hotosm.geojson
4,cvg-saint-vincent-grenadines,saint_vincent_grenadines_aoi.geojson,saint_vincent_grenadines_ref.geojson
5,dom-dominica,dominica_aoi.geojson,NaN
6,gha-aiyim-sraha,aiyim-sraha_aoi.geojson,aiyim-sraha_hotosm.geojson
7,gha-dansoman,dansoman_aoi.geojson,dansoman_hotosm.geojson
8,gha-nawuni,nawuni_aoi.geojson,nawuni_hotosm.geojson
9,gha-sawla-tuna,sawla-tuna_aoi.geojson,sawla-tuna_hotosm.geojson


In [6]:
if DRY_RUN:
    print(f"\nDry-run — {len(tracker_df)} AOIs:\n")
    for _, row in tracker_df.iterrows():
        slug = row["dataset_folder_name"].lower()
        md   = root / "outputs" / "metrics" / slug
        done = already_done(md)
        tag  = "SKIP" if (SKIP_EXISTING and done) else ("DONE" if done else "PENDING")
        print(f"  {tag:7s}  {row['dataset_folder_name']}")
else:
    print("DRY_RUN=False — proceeding to execution.")

DRY_RUN=False — proceeding to execution.


In [7]:
import traceback
import pandas as pd

if not DRY_RUN:
    results = {}

    for _, row in tracker_df.iterrows():
        folder    = row["dataset_folder_name"]
        city_slug = folder.lower()
        md        = root / "outputs" / "metrics" / city_slug

        print(f"\nProcessing {folder}...")
        if SKIP_EXISTING and already_done(md):
            print(f"[{folder}] skipped (already done)")
            results[folder] = "skipped"
            continue

        try:
            ok = process_aoi(row, cfg, root)
            results[folder] = "ok" if ok else "no_data"
        except Exception:
            print(f"[{folder}] ERROR")
            traceback.print_exc()
            results[folder] = "error"

    # Summary
    summary = pd.DataFrame(
        [(k, v) for k, v in results.items()],
        columns=["aoi", "status"]
    )
    print(f"\nDone — {len(summary)} AOIs processed.\n")
    display(summary.groupby("status")["aoi"].count().rename("count").to_frame())
    display(summary)
 

00:22:12  INFO      ━━━━  AOI: cvg-saint-vincent-grenadines  ━━━━
00:22:12  INFO      MEM [cvg-saint-vincent-grenadines start] RSS = 265 MB



Processing ant-curacao...
[ant-curacao] skipped (already done)

Processing blz-burrell-boom...
[blz-burrell-boom] skipped (already done)

Processing bra-nova-sussuarana...
[bra-nova-sussuarana] skipped (already done)

Processing col-san-antonio-de-prado...
[col-san-antonio-de-prado] skipped (already done)

Processing cvg-saint-vincent-grenadines...


00:22:13  INFO      [cvg-saint-vincent-grenadines] Auto-detected working CRS: EPSG:32620
00:22:13  INFO      [cvg-saint-vincent-grenadines] 514 tiles generated.
00:22:13  INFO      Created 514 records
00:22:18  INFO      [cvg-saint-vincent-grenadines] Reference buildings: 43061
00:22:18  WARNING   [cvg-saint-vincent-grenadines] No candidate files for dataset 'overture' (pattern: cvg_saint_vincent_grenadines_overture*.parquet in /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/data/01_raw/cvg-saint-vincent-grenadines/vector).
00:22:18  WARNING   [cvg-saint-vincent-grenadines] No candidate files for dataset 'gba' (pattern: cvg_saint_vincent_grenadines_gba*.parquet in /content/drive/MyDrive/Gates Foundation/Building Dataset Validation/data/01_raw/cvg-saint-vincent-grenadines/vector).
00:22:18  WARNING   [cvg-saint-vincent-grenadines] No candidate files for dataset 'globfp' (pattern: cvg_saint_vincent_grenadines_globfp*.parquet in /content/drive/MyDrive/Gates Foundation/


Processing dom-dominica...

Processing gha-aiyim-sraha...
[gha-aiyim-sraha] skipped (already done)

Processing gha-dansoman...
[gha-dansoman] skipped (already done)

Processing gha-nawuni...
[gha-nawuni] skipped (already done)

Processing gha-sawla-tuna...
[gha-sawla-tuna] skipped (already done)

Processing gha-wa...
[gha-wa] skipped (already done)

Processing jam-kingston...
[jam-kingston] skipped (already done)

Processing jam-saint-catherine...
[jam-saint-catherine] skipped (already done)

Processing jpn-ashiya-hama...


00:22:19  INFO      Created 1 records
00:22:19  INFO      [jpn-ashiya-hama] Reference buildings: 309
00:22:19  INFO      [jpn-ashiya-hama / overture] Loading candidate: jpn_ashiya_hama_overture_building.parquet
00:22:19  INFO      [jpn-ashiya-hama / overture] Candidate buildings: 315
00:22:20  INFO      [jpn-ashiya-hama / overture] Saved tile metrics → vector_metrics_tiles_overture.parquet
00:22:20  INFO      MEM [jpn-ashiya-hama/overture done] RSS = 347 MB
00:22:20  INFO      [jpn-ashiya-hama / gba] Loading candidate: jpn_ashiya_hama_gba.parquet
00:22:20  INFO      [jpn-ashiya-hama / gba] Candidate buildings: 0
00:22:20  INFO      [jpn-ashiya-hama / gba] Saved tile metrics → vector_metrics_tiles_gba.parquet
00:22:20  INFO      MEM [jpn-ashiya-hama/gba done] RSS = 347 MB
00:22:20  INFO      [jpn-ashiya-hama / globfp] Loading candidate: jpn_ashiya_hama_globfp.parquet
00:22:21  INFO      [jpn-ashiya-hama / globfp] Candidate buildings: 0
00:22:21  INFO      [jpn-ashiya-hama / globfp] Save

...


Processing tto-la-brea...


00:25:13  INFO      Created 1 records
00:25:13  INFO      [tto-la-brea] Reference buildings: 436
00:25:13  INFO      [tto-la-brea / overture] Loading candidate: tto_la_brea_overture_building.parquet
00:25:13  INFO      [tto-la-brea / overture] Candidate buildings: 486
00:25:15  INFO      [tto-la-brea / overture] Saved tile metrics → vector_metrics_tiles_overture.parquet
00:25:15  INFO      MEM [tto-la-brea/overture done] RSS = 433 MB
00:25:15  INFO      [tto-la-brea / gba] Loading candidate: tto_la_brea_gba.parquet
00:25:15  INFO      [tto-la-brea / gba] Candidate buildings: 580
00:25:16  INFO      [tto-la-brea / gba] Saved tile metrics → vector_metrics_tiles_gba.parquet
00:25:16  INFO      MEM [tto-la-brea/gba done] RSS = 433 MB
00:25:16  INFO      [tto-la-brea / globfp] Loading candidate: tto_la_brea_globfp.parquet
00:25:16  INFO      [tto-la-brea / globfp] Candidate buildings: 0
00:25:16  INFO      [tto-la-brea / globfp] Saved tile metrics → vector_metrics_tiles_globfp.parquet
00:25


Processing uga-bugoye...
[uga-bugoye] skipped (already done)

Processing uga-nakamiro...
[uga-nakamiro] skipped (already done)

Processing ukr-pulyny...
[ukr-pulyny] skipped (already done)

Done — 54 AOIs processed.



,count
status,
no_data,2
ok,9
skipped,43


,aoi,status
0,ant-curacao,skipped
1,blz-burrell-boom,skipped
2,bra-nova-sussuarana,skipped
3,col-san-antonio-de-prado,skipped
4,cvg-saint-vincent-grenadines,no_data
5,dom-dominica,no_data
6,gha-aiyim-sraha,skipped
7,gha-dansoman,skipped
8,gha-nawuni,skipped
9,gha-sawla-tuna,skipped
